# Experiment 1 — Does distant-supervision data augmentation help BABE news-bias detection?

This notebook performs **only the missing experiment**. It does not repeat TF-IDF baselines.

**Comparison**
- A: RoBERTa trained on BABE training sentences only.
- B: the same RoBERTa setup trained on BABE plus a controlled sample from MBIB's Wikipedia-derived sources.
- Both models use the same BABE validation and test examples. Select checkpoints using BABE validation macro-F1.

**Important:** Wikipedia neutrality / NPOV labels are not identical to expert judgments of news bias. This is an experiment in transfer from distant supervision, not a claim that these labels are interchangeable. The final result may improve, stay the same, or get worse.

**Run in Colab with a T4 GPU.** Upload `babe_cleaned.csv` if you do not already have it in the runtime.

## 1. Install and import

In [ ]:
%pip install -q transformers datasets accelerate scikit-learn pandas==2.2.3

In [ ]:
import os, random, json, re
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding,
                          EarlyStoppingCallback)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))


## 2. Reuse the existing BABE split when available

In [ ]:
DATA_PATH = Path("/content/babe_cleaned.csv")
if not DATA_PATH.exists(): DATA_PATH = Path("babe_cleaned.csv")
assert DATA_PATH.exists(), "Upload babe_cleaned.csv."

# Prefer the exact split files saved by roberta.ipynb.
SPLIT_DIR = Path("/content/babe_sg2_splits")
if not (SPLIT_DIR / "train.csv").exists():
    SPLIT_DIR = Path("babe_sg2_splits")

def normalize_label(value):
    if pd.isna(value): return np.nan
    if isinstance(value, (int, np.integer, float, np.floating)) and value in (0, 1): return int(value)
    s = str(value).strip().lower().replace("_", "-")
    return {"non-biased":0, "non biased":0, "nonbiased":0, "unbiased":0, "0":0,
            "biased":1, "1":1}.get(s, np.nan)

if all((SPLIT_DIR / f"{s}.csv").exists() for s in ["train","validation","test"]):
    train_df = pd.read_csv(SPLIT_DIR / "train.csv")
    val_df = pd.read_csv(SPLIT_DIR / "validation.csv")
    test_df = pd.read_csv(SPLIT_DIR / "test.csv")
    for part in [train_df, val_df, test_df]:
        if "label" not in part.columns: part["label"] = part["label_bias"].map(normalize_label)
        part["label"] = part["label"].astype(int)
    print("Loaded saved split files from", SPLIT_DIR)
else:
    print("Saved split files not found; reproducing the deterministic split from roberta.ipynb.")
    df = pd.read_csv(DATA_PATH)
    df["label"] = df["label_bias"].map(normalize_label)
    df["text"] = df["text"].astype("string").str.strip()
    df = df.dropna(subset=["text","news_link","label"]).copy()
    df = df[df["text"].str.len().gt(0)].copy()
    df["label"] = df["label"].astype(int)
    df["news_link"] = df["news_link"].astype(str).str.strip()
    df = df[~df["news_link"].isin(["","nan","None"])].copy()
    conflict = df.groupby("text")["label"].nunique()
    df = df[~df["text"].isin(conflict[conflict > 1].index)].drop_duplicates("text").reset_index(drop=True)
    df["article_group"] = df["news_link"]
    s1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
    trv_i, te_i = next(s1.split(df, y=df.label, groups=df.article_group))
    trv, test_df = df.iloc[trv_i].copy(), df.iloc[te_i].copy()
    s2 = GroupShuffleSplit(n_splits=1, test_size=0.17647, random_state=43)
    tr_i, va_i = next(s2.split(trv, y=trv.label, groups=trv.article_group))
    train_df, val_df = trv.iloc[tr_i].copy(), trv.iloc[va_i].copy()
    for part in [train_df,val_df,test_df]:
        part.drop(columns=["article_group"], errors="ignore", inplace=True)
    SPLIT_DIR.mkdir(parents=True, exist_ok=True)
    for name,part in [("train",train_df),("validation",val_df),("test",test_df)]:
        part.to_csv(SPLIT_DIR/f"{name}.csv", index=False)

for name,part in [("train",train_df),("validation",val_df),("test",test_df)]:
    print(name, len(part), "labels", part.label.value_counts().sort_index().to_dict(),
          "articles", part.news_link.nunique() if "news_link" in part else "n/a")
assert set(train_df.news_link).isdisjoint(set(val_df.news_link))
assert set(train_df.news_link).isdisjoint(set(test_df.news_link))
assert set(val_df.news_link).isdisjoint(set(test_df.news_link))


## 3. Load and audit the Wikipedia-derived MBIB examples

In [ ]:
# MBIB's linguistic_bias configuration includes multiple datasets with different label meanings.
# Start with dataset_id 3 (Wikipedia NPOV) and 26 (Wiki Neutrality Corpus), not all sources.
# If the repository schema changes, inspect ds.features and ds[0] before proceeding.
ds = load_dataset("mediabiasgroup/mbib-base", "linguistic_bias", split="train")
mbib = ds.to_pandas()
print("Columns:", mbib.columns.tolist())
print("Rows:", len(mbib))
display(mbib.head())
assert {"text","label","dataset_id"}.issubset(mbib.columns), "Schema changed; inspect dataset fields."
print("Counts by source and label:")
display(pd.crosstab(mbib["dataset_id"], mbib["label"], margins=True))


## 4. Create a conservative, balanced augmentation sample

In [ ]:
# Use only the two Wikipedia-derived sources identified in the MBIB repository:
# 3 = WikiNPOV; 26 = WikiNeutralityCorpus.
# We sample a modest amount to keep training feasible on a T4.
# This is an explicit experimental choice, not an assumption that these labels are perfect.

wiki = mbib[mbib["dataset_id"].astype(str).isin(["3","26"])].copy()
wiki["text"] = wiki["text"].astype(str).str.strip()
wiki["label"] = pd.to_numeric(wiki["label"], errors="coerce")
wiki = wiki.dropna(subset=["text","label"])
wiki = wiki[wiki["label"].isin([0,1])].copy()
wiki["label"] = wiki["label"].astype(int)
wiki = wiki[wiki["text"].str.len().between(20, 1500)].copy()
wiki["text_norm"] = wiki["text"].str.lower().str.replace(r"\s+"," ",regex=True).str.strip()
wiki = wiki.drop_duplicates("text_norm")

# Prevent exact text overlap with any BABE split.
babe_texts = set(pd.concat([train_df.text,val_df.text,test_df.text]).astype(str)
                 .str.lower().str.replace(r"\s+"," ",regex=True).str.strip())
wiki = wiki[~wiki["text_norm"].isin(babe_texts)]
print("Wikipedia-derived rows after cleaning and BABE exact-text dedup:", len(wiki))
display(pd.crosstab(wiki["dataset_id"], wiki["label"]))

# Balanced sample per source/label where possible, up to 2,000 rows per label in total.
# Keep source ID for audit, then remove it from model input.
MAX_PER_LABEL = 2000
sample_parts=[]
for label in [0,1]:
    part = wiki[wiki.label == label]
    take = min(MAX_PER_LABEL, len(part))
    sample_parts.append(part.sample(n=take, random_state=SEED + label))
wiki_sample = pd.concat(sample_parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
print("Augmentation sample:", len(wiki_sample))
print("Label counts:", wiki_sample.label.value_counts().sort_index().to_dict())
display(wiki_sample.groupby(["dataset_id","label"]).size().rename("n").reset_index())


## 5. Train both conditions with the same model settings

In [ ]:
MODEL_NAME = "FacebookAI/roberta-base"
MAX_LENGTH = 256
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def to_hf(frame):
    d = Dataset.from_pandas(frame[["text","label"]].copy().reset_index(drop=True), preserve_index=False)
    def tok(batch): return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)
    return d.map(tok, batched=True, remove_columns=["text"])

train_babe_hf = to_hf(train_df)
train_aug_hf = to_hf(pd.concat([
    train_df[["text","label"]],
    wiki_sample[["text","label"]]
], ignore_index=True))
val_hf = to_hf(val_df)
test_hf = to_hf(test_df)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels,pred),
            "macro_f1": f1_score(labels,pred,average="macro",zero_division=0)}

def train_condition(condition, train_hf):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, id2label={0:"Non-biased",1:"Biased"},
        label2id={"Non-biased":0,"Biased":1})
    outdir = f"/content/augmentation_experiment_{condition}"
    args = TrainingArguments(
        output_dir=outdir, learning_rate=1e-5,
        per_device_train_batch_size=8, per_device_eval_batch_size=16,
        num_train_epochs=3, weight_decay=0.01,
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="eval_macro_f1",
        greater_is_better=True, fp16=torch.cuda.is_available(),
        logging_steps=50, report_to="none", seed=SEED, data_seed=SEED,
        save_total_limit=1)
    trainer = Trainer(model=model,args=args,train_dataset=train_hf,
        eval_dataset=val_hf,data_collator=collator,compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)])
    trainer.train()
    val_metrics = trainer.evaluate(val_hf)
    # Test is evaluated only after checkpoint selection is complete.
    test_out = trainer.predict(test_hf)
    pred = np.argmax(test_out.predictions, axis=-1)
    result = {
        "condition":condition,
        "best_validation_macro_f1":float(trainer.state.best_metric),
        "test_accuracy":float(accuracy_score(test_out.label_ids,pred)),
        "test_macro_f1":float(f1_score(test_out.label_ids,pred,average="macro",zero_division=0)),
        "test_biased_f1":float(f1_score(test_out.label_ids,pred,pos_label=1,zero_division=0)),
        "test_nonbiased_f1":float(f1_score(test_out.label_ids,pred,pos_label=0,zero_division=0)),
        "test_predictions":pred.tolist()
    }
    trainer.save_model(outdir + "/best_model")
    return result, trainer

# Two training runs; on a T4 this can take a while.
baseline_result, baseline_trainer = train_condition("babe_only", train_babe_hf)
augmented_result, augmented_trainer = train_condition("babe_plus_wikipedia", train_aug_hf)


## 6. Compare results and inspect error types

In [ ]:
results = pd.DataFrame([
    {k:v for k,v in baseline_result.items() if k != "test_predictions"},
    {k:v for k,v in augmented_result.items() if k != "test_predictions"}
])
display(results)
print("Change in test macro-F1 (augmented - BABE only):",
      round(augmented_result["test_macro_f1"] - baseline_result["test_macro_f1"], 4))

y_test = test_df["label"].to_numpy()
for result in [baseline_result, augmented_result]:
    print("\\n###", result["condition"])
    print(classification_report(y_test, result["test_predictions"],
          labels=[0,1], target_names=["Non-biased","Biased"], digits=4, zero_division=0))
    print("Confusion matrix:\\n", confusion_matrix(y_test,result["test_predictions"]))


## 7. Save the experiment artefacts

In [ ]:
OUT = Path("/content/newsbias_augmentation_results")
OUT.mkdir(parents=True, exist_ok=True)
results.to_csv(OUT/"comparison_metrics.csv", index=False)
preds = test_df[["text","label","news_link"]].reset_index(drop=True).copy()
preds["pred_babe_only"] = baseline_result["test_predictions"]
preds["pred_babe_plus_wikipedia"] = augmented_result["test_predictions"]
preds["baseline_correct"] = preds.label == preds.pred_babe_only
preds["augmented_correct"] = preds.label == preds.pred_babe_plus_wikipedia
preds.to_csv(OUT/"test_predictions_comparison.csv", index=False)
wiki_sample.drop(columns=["text_norm"],errors="ignore").to_csv(OUT/"wiki_augmentation_sample.csv",index=False)
with open(OUT/"experiment_summary.json","w") as f:
    json.dump({
        "seed":SEED,"model":MODEL_NAME,"max_length":MAX_LENGTH,
        "wiki_sample_size":len(wiki_sample),
        "wiki_source_ids":sorted(wiki_sample.dataset_id.astype(str).unique().tolist()),
        "baseline":{k:v for k,v in baseline_result.items() if k!="test_predictions"},
        "augmented":{k:v for k,v in augmented_result.items() if k!="test_predictions"},
        "caveat":"Wikipedia-derived labels are distant supervision and may not match expert-annotated news bias."
    },f,indent=2)
print("Saved to", OUT)
display(preds[(preds.baseline_correct != preds.augmented_correct)].head(25))


## Interpretation checklist for the paper

- Did augmentation improve **validation macro-F1** and final test macro-F1, or only one of them?
- Which class changed most? Did biased recall improve at the cost of non-biased precision?
- Did source augmentation change error patterns?
- Do not claim causality beyond this controlled comparison, and do not claim novelty until related work has been checked.
- This notebook evaluates the same BABE test set for both conditions. If that test set has already been repeatedly used to guide model choices, disclose this limitation or create a new locked article-disjoint test set for a final study.
- SHAP is a follow-up analysis, not the primary novelty claim; the key experiment here is controlled distant-supervision augmentation.